# Free Speed: Same E4B Text, 3.6x Faster Decode on a Laptop: reproduce every number

CPU only, no model, no competition data. The embedded table has one row per (experiment, model, decoding config, prompt): token counts, decode milliseconds and a SHA-256 of the model output. The cells below rebuild the paper's main table and check it against the paper's numbers.


In [ ]:
import base64, gzip, io, csv, json
BLOB = '''...'''
rows = list(csv.DictReader(io.StringIO(gzip.decompress(base64.b64decode(BLOB)).decode())))
PAPER = json.loads('''{"W1.12b.baseline_rerun.ALL.identity_rate": 1.0, "W1.12b.baseline_rerun.ALL.n_pairs": 27, "W1.12b.baseline_rerun.ALL.speedup": 1.4639847558145394, "W1.12b.mtp.ALL.identity_rate": 0.8333333333333334, "W1.12b.mtp.ALL.n_pairs": 24, "W1.12b.mtp.ALL.speedup": 1.2560748270356878, "W1.12b.mtp_ngram.ALL.identity_rate": 0.84375, "W1.12b.mtp_ngram.ALL.n_pairs": 32, "W1.12b.mtp_ngram.ALL.speedup": 1.6542251302308466, "W1.12b.ngram.ALL.identity_rate": 0.96875, "W1.12b.ngram.ALL.n_pairs": 32, "W1.12b.ngram.ALL.speedup": 1.2363459653464453, "W1.e4b.mtp.ALL.identity_rate": 0.9166666666666666, "W1.e4b.mtp.ALL.n_pairs": 24, "W1.e4b.mtp.ALL.speedup": 3.0776508375791756, "W1.e4b.mtp_ngram.ALL.identity_rate": 0.875, "W1.e4b.mtp_ngram.ALL.n_pairs": 24, "W1.e4b.mtp_ngram.ALL.speedup": 2.337368362316538, "W1.e4b.ngram.ALL.identity_rate": 0.875, "W1.e4b.ngram.ALL.n_pairs": 24, "W1.e4b.ngram.ALL.speedup": 1.8263248642345287, "W1full.e4b.baseline_rerun.ALL.identity_rate": 1.0, "W1full.e4b.baseline_rerun.ALL.n_pairs": 24, "W1full.e4b.baseline_rerun.ALL.speedup": 1.100574099461045, "W1full.e4b.mtp.ALL.identity_rate": 1.0, "W1full.e4b.mtp.ALL.n_pairs": 24, "W1full.e4b.mtp.ALL.speedup": 3.380751902293788, "W1full.e4b.mtp_ngram.ALL.identity_rate": 1.0, "W1full.e4b.mtp_ngram.ALL.n_pairs": 24, "W1full.e4b.mtp_ngram.ALL.speedup": 3.9661974186201805, "W1s.12b.ngram.ALL.identity_rate": 1.0, "W1s.12b.ngram.ALL.n_pairs": 10, "W1s.12b.ngram.ALL.speedup": 3.266522023201709, "W2.12b.mtp.ALL.identity_rate": 0.4, "W2.12b.mtp.ALL.n_pairs": 30, "W2.12b.mtp.ALL.speedup": 2.5035320104003493, "W2.12b.mtp_ngram.ALL.identity_rate": 0.4, "W2.12b.mtp_ngram.ALL.n_pairs": 30, "W2.12b.mtp_ngram.ALL.speedup": 1.5814923327356534, "W2.12b.ngram.ALL.identity_rate": 0.6, "W2.12b.ngram.ALL.n_pairs": 30, "W2.12b.ngram.ALL.speedup": 1.8969068177996813}''')
print(len(rows), 'records')


In [ ]:
base = {(r["tag"], r["target"], r["id"]): r for r in rows if r["config"] == "baseline"}
groups = {}
for r in rows:
    if r["config"] == "baseline":
        continue
    b = base.get((r["tag"], r["target"], r["id"]))
    if b:
        groups.setdefault((r["tag"], r["target"], r["config"]), []).append((b, r))
table, bad = [], 0
for (tag, target, config), P in sorted(groups.items()):
    n = len(P)
    ident = sum(b["text_sha256"] == r["text_sha256"] for b, r in P) / n
    bn = sum(float(b["predicted_n"]) for b, r in P); bms = sum(float(b["predicted_ms"]) for b, r in P)
    cn = sum(float(r["predicted_n"]) for b, r in P); cms = sum(float(r["predicted_ms"]) for b, r in P)
    sp = (cn / cms) / (bn / bms)
    table.append((tag, target, config, n, ident, sp))
    key = "%s.%s.%s.ALL." % (tag, target, config)
    for name, val, tol in (("speedup", sp, 0.01), ("identity_rate", ident, 1e-9), ("n_pairs", n, 0)):
        if key + name in PAPER and abs(PAPER[key + name] - val) > tol:
            bad += 1; print("MISMATCH", key + name, PAPER[key + name], val)
print("%-8s %-5s %-16s %5s %10s %8s" % ("tag", "model", "config", "n", "identical", "speedup"))
for t in table:
    print("%-8s %-5s %-16s %5d %9.1f%% %7.2fx" % (t[0], t[1], t[2], t[3], 100 * t[4], t[5]))
print("ALL PAPER NUMBERS REPRODUCED" if bad == 0 else "%d MISMATCHES" % bad)


In [ ]:
import matplotlib.pyplot as plt
show = [t for t in table if not t[2].startswith("baseline")]
plt.figure(figsize=(max(6, len(show)), 4))
plt.bar(range(len(show)), [t[5] for t in show]); plt.axhline(1, color="k", ls="--")
plt.xticks(range(len(show)), ["%s\n%s\n%s" % t[:3] for t in show], fontsize=7)
plt.ylabel("Decode speedup vs baseline"); plt.tight_layout(); plt.show()
